In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
# --- Silver bootstrap: redefine everything in case session restarted ---

import notebookutils
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, LongType, DoubleType,
    StringType, TimestampType, TimestampNTZType
)
from datetime import datetime, timezone

# Lakehouse paths
BRONZE_LH = notebookutils.lakehouse.getWithProperties("bronze")["properties"]["abfsPath"]
SILVER_LH = notebookutils.lakehouse.getWithProperties("silver")["properties"]["abfsPath"]

BRONZE_TRIPS = f"{BRONZE_LH}/Tables/yellow_tripdata"
BRONZE_ZONES = f"{BRONZE_LH}/Tables/taxi_zone_lookup"
SILVER_TRIPS = f"{SILVER_LH}/Tables/yellow_tripdata"
SILVER_QUARANTINE = f"{SILVER_LH}/Tables/yellow_tripdata_quarantine"
SILVER_ZONES = f"{SILVER_LH}/Tables/taxi_zone_lookup"

# Valid date window for our slice
VALID_FROM = "2024-01-01"
VALID_TO = "2024-07-01"

processing_ts = datetime.now(timezone.utc)

print(f"Bronze trips: {BRONZE_TRIPS}")
print(f"Silver trips: {SILVER_TRIPS}")
print(f"Silver quarantine: {SILVER_QUARANTINE}")
print(f"Processing timestamp: {processing_ts.isoformat()}")

StatementMeta(, a271f6b8-2191-4db4-995b-e2238ecb36c8, 3, Finished, Available, Finished, False)

Bronze trips: abfss://73a7dd19-3e75-431d-9f13-0dbc0f72f9c3@onelake.dfs.fabric.microsoft.com/6c7e6744-c4e8-4ea3-b24e-2fcf234dc09e/Tables/yellow_tripdata
Silver trips: abfss://73a7dd19-3e75-431d-9f13-0dbc0f72f9c3@onelake.dfs.fabric.microsoft.com/4186ae61-ab44-4157-bcc5-a94295cd9cf6/Tables/yellow_tripdata
Silver quarantine: abfss://73a7dd19-3e75-431d-9f13-0dbc0f72f9c3@onelake.dfs.fabric.microsoft.com/4186ae61-ab44-4157-bcc5-a94295cd9cf6/Tables/yellow_tripdata_quarantine
Processing timestamp: 2026-05-27T19:30:47.518384+00:00


In [2]:
# --- Silver Step 1: Read Bronze + apply quarantine rules + standardize ---

# Read bronze tables via ABFSS (avoids friendly-name issues)
bronze_df = spark.read.format("delta").load(BRONZE_TRIPS)
zones_df = spark.read.format("delta").load(BRONZE_ZONES)

print(f"Bronze rows read: {bronze_df.count():,}")
print(f"Zone rows read: {zones_df.count()}")

# --- Quarantine flag logic ---
# Mark rows that fail any data-quality rule. We keep ALL rows and split them later
# so we don't silently drop data and so we can reconcile back to bronze row counts.

tagged_df = (
    bronze_df
    # Derived: trip duration in seconds
    .withColumn(
        "trip_duration_seconds",
        F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")
    )
    # Quarantine flags
    .withColumn("_q_outside_window",
        (F.col("tpep_pickup_datetime") < F.to_timestamp(F.lit(VALID_FROM))) |
        (F.col("tpep_pickup_datetime") >= F.to_timestamp(F.lit(VALID_TO)))
    )
    .withColumn("_q_negative_duration", F.col("trip_duration_seconds") < 0)
    .withColumn("_q_excessive_duration", F.col("trip_duration_seconds") > 86400)  # >24hr
    .withColumn("_q_invalid_distance",
        (F.col("trip_distance") < 0) | (F.col("trip_distance") > 200)
    )
    .withColumn("_q_negative_total", F.col("total_amount") < 0)
    # Combined quarantine flag — any rule triggered
    .withColumn("_quarantine",
        F.col("_q_outside_window") |
        F.col("_q_negative_duration") |
        F.col("_q_excessive_duration") |
        F.col("_q_invalid_distance") |
        F.col("_q_negative_total")
    )
)

# Show the breakdown of why rows get quarantined
print("\n=== Quarantine reasons breakdown ===")
quarantine_breakdown = tagged_df.filter(F.col("_quarantine")).agg(
    F.sum(F.col("_q_outside_window").cast("int")).alias("outside_window"),
    F.sum(F.col("_q_negative_duration").cast("int")).alias("negative_duration"),
    F.sum(F.col("_q_excessive_duration").cast("int")).alias("excessive_duration"),
    F.sum(F.col("_q_invalid_distance").cast("int")).alias("invalid_distance"),
    F.sum(F.col("_q_negative_total").cast("int")).alias("negative_total"),
    F.count("*").alias("total_quarantined")
)
quarantine_breakdown.show(truncate=False)

# Cache because we'll filter into two writes
tagged_df.cache()
total_rows = tagged_df.count()
quarantined_rows = tagged_df.filter(F.col("_quarantine")).count()
clean_rows = total_rows - quarantined_rows

print(f"\nTotal rows: {total_rows:,}")
print(f"Clean rows: {clean_rows:,} ({100 * clean_rows / total_rows:.4f}%)")
print(f"Quarantined rows: {quarantined_rows:,} ({100 * quarantined_rows / total_rows:.4f}%)")

StatementMeta(, a271f6b8-2191-4db4-995b-e2238ecb36c8, 4, Finished, Available, Finished, False)

Bronze rows read: 20,332,093
Zone rows read: 265

=== Quarantine reasons breakdown ===
+--------------+-----------------+------------------+----------------+--------------+-----------------+
|outside_window|negative_duration|excessive_duration|invalid_distance|negative_total|total_quarantined|
+--------------+-----------------+------------------+----------------+--------------+-----------------+
|39            |300              |109               |539             |257431        |258409           |
+--------------+-----------------+------------------+----------------+--------------+-----------------+


Total rows: 20,332,093
Clean rows: 20,073,684 (98.7291%)
Quarantined rows: 258,409 (1.2709%)


In [3]:
# --- Silver Step 2: Standardize + write both tables ---

# Vendor and payment-type lookups (TLC publishes these in their data dictionary)
VENDORS = spark.createDataFrame([
    (1, "Creative Mobile Technologies"),
    (2, "VeriFone Inc."),
    (6, "Myle Technologies"),
    (7, "Helix"),
], ["VendorID", "vendor_name"])

PAYMENT_TYPES = spark.createDataFrame([
    (0, "Flex Fare"),
    (1, "Credit card"),
    (2, "Cash"),
    (3, "No charge"),
    (4, "Dispute"),
    (5, "Unknown"),
    (6, "Voided trip"),
], ["payment_type", "payment_type_name"])

# Slim zone columns for join
zones_slim = zones_df.select(
    F.col("LocationID").alias("zone_LocationID"),
    F.col("Borough").alias("zone_Borough"),
    F.col("Zone").alias("zone_Zone"),
    F.col("service_zone").alias("zone_service_zone"),
)

# Helper to standardize: types, joins, audit columns
def standardize(df):
    return (
        df
        # Cast columns where Bronze inference was wider than needed
        .withColumn("passenger_count", F.col("passenger_count").cast(IntegerType()))
        .withColumn("RatecodeID", F.col("RatecodeID").cast(IntegerType()))
        .withColumn("payment_type", F.col("payment_type").cast(IntegerType()))
        # Trim string fields
        .withColumn("store_and_fwd_flag", F.trim(F.col("store_and_fwd_flag")))
        # Vendor name lookup
        .join(VENDORS, on="VendorID", how="left")
        # Payment type name lookup
        .join(PAYMENT_TYPES, on="payment_type", how="left")
        # Pickup zone enrichment
        .join(
            zones_slim.alias("pu"),
            F.col("PULocationID") == F.col("pu.zone_LocationID"),
            "left"
        )
        .withColumnRenamed("zone_Borough", "PU_Borough")
        .withColumnRenamed("zone_Zone", "PU_Zone")
        .withColumnRenamed("zone_service_zone", "PU_service_zone")
        .drop("zone_LocationID")
        # Dropoff zone enrichment
        .join(
            zones_slim.alias("do"),
            F.col("DOLocationID") == F.col("do.zone_LocationID"),
            "left"
        )
        .withColumnRenamed("zone_Borough", "DO_Borough")
        .withColumnRenamed("zone_Zone", "DO_Zone")
        .withColumnRenamed("zone_service_zone", "DO_service_zone")
        .drop("zone_LocationID")
        # Audit column
        .withColumn("_silver_processed_at", F.lit(processing_ts).cast(TimestampType()))
    )


# --- Write quarantine table (small, all flags preserved for inspection) ---
quarantine_df = standardize(tagged_df.filter(F.col("_quarantine")))

(
    quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_QUARANTINE)
)
print(f"✓ Wrote quarantine: {quarantine_df.count():,} rows → {SILVER_QUARANTINE}")


# --- Write Silver table (clean rows only; drop the quarantine flags) ---
silver_df = (
    standardize(tagged_df.filter(~F.col("_quarantine")))
    .drop("_quarantine", "_q_outside_window", "_q_negative_duration",
          "_q_excessive_duration", "_q_invalid_distance", "_q_negative_total")
)

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("_pickup_year", "_pickup_month")
    .save(SILVER_TRIPS)
)
print(f"✓ Wrote silver: {silver_df.count():,} rows → {SILVER_TRIPS}")


# --- Write Silver zone lookup (just a passthrough with audit column) ---
silver_zones = zones_df.withColumn(
    "_silver_processed_at", F.lit(processing_ts).cast(TimestampType())
)
(
    silver_zones.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_ZONES)
)
print(f"✓ Wrote silver zones: {silver_zones.count()} rows → {SILVER_ZONES}")

StatementMeta(, a271f6b8-2191-4db4-995b-e2238ecb36c8, 5, Finished, Available, Finished, False)

✓ Wrote quarantine: 258,409 rows → abfss://73a7dd19-3e75-431d-9f13-0dbc0f72f9c3@onelake.dfs.fabric.microsoft.com/4186ae61-ab44-4157-bcc5-a94295cd9cf6/Tables/yellow_tripdata_quarantine
✓ Wrote silver: 20,073,684 rows → abfss://73a7dd19-3e75-431d-9f13-0dbc0f72f9c3@onelake.dfs.fabric.microsoft.com/4186ae61-ab44-4157-bcc5-a94295cd9cf6/Tables/yellow_tripdata
✓ Wrote silver zones: 265 rows → abfss://73a7dd19-3e75-431d-9f13-0dbc0f72f9c3@onelake.dfs.fabric.microsoft.com/4186ae61-ab44-4157-bcc5-a94295cd9cf6/Tables/taxi_zone_lookup


In [4]:
# --- Silver Step 3: Validation ---

silver = spark.read.format("delta").load(SILVER_TRIPS)
quarantine = spark.read.format("delta").load(SILVER_QUARANTINE)

print("=== Check 1: row count reconciliation ===")
total_silver = silver.count()
total_quarantine = quarantine.count()
print(f"Silver clean:     {total_silver:>12,}")
print(f"Silver quarantine:{total_quarantine:>12,}")
print(f"Total:            {total_silver + total_quarantine:>12,}  (should equal bronze: 20,332,093)")

print("\n=== Check 2: derived columns populated ===")
silver.select(
    F.min("trip_duration_seconds").alias("min_duration"),
    F.max("trip_duration_seconds").alias("max_duration"),
    F.avg("trip_duration_seconds").alias("avg_duration"),
).show()

print("=== Check 3: standardization joins worked (no nulls in vendor/payment/zone names) ===")
silver.select(
    F.count(F.when(F.col("vendor_name").isNull(), 1)).alias("null_vendor_name"),
    F.count(F.when(F.col("payment_type_name").isNull(), 1)).alias("null_payment_name"),
    F.count(F.when(F.col("PU_Borough").isNull(), 1)).alias("null_pu_borough"),
    F.count(F.when(F.col("DO_Borough").isNull(), 1)).alias("null_do_borough"),
).show()

print("=== Check 4: top 5 pickup boroughs ===")
silver.groupBy("PU_Borough").count().orderBy(F.desc("count")).show(5, truncate=False)

print("=== Check 5: payment type distribution ===")
silver.groupBy("payment_type_name").count().orderBy(F.desc("count")).show(truncate=False)

print("=== Check 6: schema preview ===")
silver.printSchema()

StatementMeta(, a271f6b8-2191-4db4-995b-e2238ecb36c8, 6, Finished, Available, Finished, False)

=== Check 1: row count reconciliation ===
Silver clean:       20,073,684
Silver quarantine:     258,409
Total:              20,332,093  (should equal bronze: 20,332,093)

=== Check 2: derived columns populated ===
+------------+------------+------------------+
|min_duration|max_duration|      avg_duration|
+------------+------------+------------------+
|           0|       86399|1014.4083161317076|
+------------+------------+------------------+

=== Check 3: standardization joins worked (no nulls in vendor/payment/zone names) ===
+----------------+-----------------+---------------+---------------+
|null_vendor_name|null_payment_name|null_pu_borough|null_do_borough|
+----------------+-----------------+---------------+---------------+
|               0|                0|              0|              0|
+----------------+-----------------+---------------+---------------+

=== Check 4: top 5 pickup boroughs ===
+----------+--------+
|PU_Borough|count   |
+----------+--------+
|Manhattan |1